# Build Final Embeddings

This notebook builds combined embeddings for all selected late-fusion model runs.

Main export:
- `train + validation`

Separate bottom section:
- `train + validation + test`

Everything is saved under `data/processed/model_runs` in clearly named folders.

## Main Goal

For each selected model run, save a combined `train_validation` embedding set into:

`data/processed/model_runs/final_embeddings_train_validation/<run_label>/`

At the very bottom, there is a separate optional section to also build:

`data/processed/model_runs/final_embeddings_train_validation_test/<run_label>/`

In [7]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.models import resnet18

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists() and ROOT_DIR.parent.exists():
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from song_recommender.paths import DATA_DIR
from song_recommender.data.dataset import StemSongDataset

In [8]:
# =========================
# Config (edit this cell)
# =========================
CFG = {
    "run_labels": [
        "04_Resnet18_contrastive_tags",
        "05_Resnet18_contrastive_tags_infonce",
        "06_Resnet18_contrastive_tags_audio_grounded_infonce",
        "07_Resnet18_audio_centric_blended_teacher",
    ],
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "split_paths": {
        "train": DATA_DIR / "processed" / "train.parquet",
        "validation": DATA_DIR / "processed" / "val.parquet",
        "test": DATA_DIR / "processed" / "test.parquet",
    },
    "train_validation_root": DATA_DIR / "processed" / "model_runs" / "final_embeddings_train_validation",
    "train_validation_test_root": DATA_DIR / "processed" / "model_runs" / "final_embeddings_train_validation_test",
}

CFG["train_validation_root"].mkdir(parents=True, exist_ok=True)
CFG["train_validation_test_root"].mkdir(parents=True, exist_ok=True)

print("models:", CFG["run_labels"])
print("train_validation_root:", CFG["train_validation_root"])
print("train_validation_test_root:", CFG["train_validation_test_root"])
print("device:", CFG["device"])

models: ['04_Resnet18_contrastive_tags', '05_Resnet18_contrastive_tags_infonce', '06_Resnet18_contrastive_tags_audio_grounded_infonce', '07_Resnet18_audio_centric_blended_teacher']
train_validation_root: C:\Users\Juan\Documents\GitHub\dl-song-recommender\data\processed\model_runs\final_embeddings_train_validation
train_validation_test_root: C:\Users\Juan\Documents\GitHub\dl-song-recommender\data\processed\model_runs\final_embeddings_train_validation_test
device: cuda


In [9]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim: int, projection_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(inplace=True),
            nn.Linear(input_dim, projection_dim),
        )

    def forward(self, x):
        return self.net(x)


class StemLateFusionResNet18(nn.Module):
    def __init__(
        self,
        embedding_dim=64,
        projection_dim=64,
        pretrained=False,
        imagenet_input_norm=None,
        fusion_alpha_init=0.7,
        drum_alpha_init=0.3,
        num_stems=4,
        stem_dropout_prob=0.1,
        harmonic_indices=(0, 2, 3),
        drum_index=1,
    ):
        super().__init__()
        backbone = resnet18(weights=None)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Linear(in_features, embedding_dim)

        self.encoder = backbone
        self.projection_head = ProjectionHead(embedding_dim, projection_dim)
        if imagenet_input_norm is None:
            imagenet_input_norm = pretrained
        self.imagenet_input_norm = bool(imagenet_input_norm)
        self.register_buffer(
            "imagenet_mean",
            torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1),
            persistent=False,
        )
        self.register_buffer(
            "imagenet_std",
            torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1),
            persistent=False,
        )

        self.fusion_alpha_logit = nn.Parameter(torch.tensor(fusion_alpha_init, dtype=torch.float32).logit())
        self.drum_alpha_logit = nn.Parameter(torch.tensor(drum_alpha_init, dtype=torch.float32).logit())
        self.num_stems = int(num_stems)
        self.harmonic_indices = tuple(int(idx) for idx in harmonic_indices)
        self.drum_index = int(drum_index)
        self.stem_logits = nn.Parameter(torch.zeros(self.num_stems, dtype=torch.float32))
        self.harmonic_logits = nn.Parameter(torch.zeros(len(self.harmonic_indices), dtype=torch.float32))
        self.register_buffer(
            "harmonic_index_tensor",
            torch.tensor(self.harmonic_indices, dtype=torch.long),
            persistent=False,
        )
        self.stem_dropout_prob = float(stem_dropout_prob)

    def _encode_inputs(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        if self.imagenet_input_norm:
            x = (x - self.imagenet_mean) / self.imagenet_std
        return self.encoder(x)

    def forward(self, mix, stems):
        batch_size, num_stems, channels, height, width = stems.shape
        mix_embedding = self._encode_inputs(mix)
        stem_inputs = stems.reshape(batch_size * num_stems, channels, height, width)
        stem_embeddings = self._encode_inputs(stem_inputs).reshape(batch_size, num_stems, -1)
        stem_embeddings = F.normalize(stem_embeddings, dim=2, eps=1e-8)

        stem_weights = torch.softmax(self.stem_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
        stem_weights = stem_weights / stem_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)

        harmonic_stems = stem_embeddings[:, self.harmonic_index_tensor, :]
        harmonic_weights = torch.softmax(self.harmonic_logits, dim=0).unsqueeze(0).expand(batch_size, -1)
        harmonic_weights = harmonic_weights * stem_weights[:, self.harmonic_index_tensor]
        harmonic_weights = torch.where(
            harmonic_weights.sum(dim=1, keepdim=True) == 0,
            torch.ones_like(harmonic_weights),
            harmonic_weights,
        )
        harmonic_weights = harmonic_weights / harmonic_weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        harmonic_embedding = torch.sum(harmonic_stems * harmonic_weights.unsqueeze(-1), dim=1)
        drum_embedding = stem_embeddings[:, self.drum_index, :] * stem_weights[:, self.drum_index].unsqueeze(-1)

        mix_embedding = F.normalize(mix_embedding, dim=1, eps=1e-8)
        harmonic_embedding = F.normalize(harmonic_embedding, dim=1, eps=1e-8)
        drum_embedding = F.normalize(drum_embedding, dim=1, eps=1e-8)

        alpha_h = torch.sigmoid(self.fusion_alpha_logit)
        alpha_d = torch.sigmoid(self.drum_alpha_logit)
        song_embedding = F.normalize(
            mix_embedding + alpha_h * harmonic_embedding + alpha_d * drum_embedding,
            dim=1,
            eps=1e-8,
        )

        return {
            "song_embedding": song_embedding,
            "mix_embedding": mix_embedding,
            "fused_stem_embedding": harmonic_embedding,
        }


def load_checkpoint_model(checkpoint_path: Path, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model_init_kwargs = checkpoint.get("model_init_kwargs", {}).copy()
    if not model_init_kwargs:
        checkpoint_cfg = checkpoint.get("cfg", {})
        model_init_kwargs = {
            "embedding_dim": int(checkpoint["embedding_dim"]),
            "projection_dim": int(checkpoint["projection_dim"]),
            "pretrained": False,
            "imagenet_input_norm": bool(checkpoint_cfg.get("pretrained", False)),
        }
    else:
        model_init_kwargs["pretrained"] = False

    model = StemLateFusionResNet18(**model_init_kwargs).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint, model_init_kwargs


@torch.no_grad()
def build_song_embeddings(model, loader, device: torch.device):
    track_ids = []
    song_embeddings = []
    mix_embeddings = []
    stem_embeddings = []

    model.eval()
    for batch in loader:
        mix = batch["mix"].to(device)
        stems = batch["stems"].to(device)
        out = model(mix, stems)

        track_ids.extend(batch["track_id"])
        song_embeddings.append(F.normalize(out["song_embedding"], dim=1, eps=1e-8).cpu())
        mix_embeddings.append(F.normalize(out["mix_embedding"], dim=1, eps=1e-8).cpu())
        stem_embeddings.append(F.normalize(out["fused_stem_embedding"], dim=1, eps=1e-8).cpu())

    return {
        "spotify_id": np.asarray(track_ids, dtype=str),
        "song_embeddings": torch.cat(song_embeddings, dim=0).numpy().astype(np.float32),
        "mix_embeddings": torch.cat(mix_embeddings, dim=0).numpy().astype(np.float32),
        "stem_embeddings": torch.cat(stem_embeddings, dim=0).numpy().astype(np.float32),
    }


def concat_frames(split_names: list[str]) -> pd.DataFrame:
    frames = [pd.read_parquet(CFG["split_paths"][name]).reset_index(drop=True) for name in split_names]
    out = pd.concat(frames, axis=0, ignore_index=True)
    if out["spotify_id"].duplicated().any():
        dupes = out.loc[out["spotify_id"].duplicated(), "spotify_id"].astype(str).unique().tolist()
        raise ValueError(f"Duplicate spotify_id values found across splits: {dupes[:5]}")
    return out


def reorder_frame_to_track_ids(frame: pd.DataFrame, track_ids: np.ndarray) -> pd.DataFrame:
    ordered = frame.set_index("spotify_id").loc[track_ids].reset_index()
    if len(ordered) != len(track_ids):
        raise ValueError("Metadata row count does not match embedding row count after reordering.")
    if ordered["spotify_id"].astype(str).tolist() != track_ids.tolist():
        raise ValueError("Metadata ordering does not match embedding ordering.")
    return ordered


def export_combined_embeddings(run_label: str, split_names: list[str], output_root: Path, combined_name: str):
    run_dir = (DATA_DIR / "processed" / "model_runs" / run_label).resolve()
    checkpoint_path = run_dir / "checkpoint.pt"
    manifest_path = run_dir / "run_manifest.json"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    output_dir = (output_root / run_label).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device(CFG["device"])
    model, _, model_init_kwargs = load_checkpoint_model(checkpoint_path, device)
    source_manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}

    combined_df = concat_frames(split_names)
    dataset = StemSongDataset(combined_df, image_size=CFG["image_size"], transform=None)
    loader = DataLoader(
        dataset,
        batch_size=CFG["batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=(device.type == "cuda"),
    )

    embedding_out = build_song_embeddings(model, loader, device)
    ordered_df = reorder_frame_to_track_ids(combined_df, embedding_out["spotify_id"])

    npz_path = output_dir / "embeddings.npz"
    metadata_path = output_dir / "metadata.parquet"
    manifest_out_path = output_dir / "manifest.json"

    np.savez(
        npz_path,
        spotify_id=embedding_out["spotify_id"],
        song_embeddings=embedding_out["song_embeddings"],
        mix_embeddings=embedding_out["mix_embeddings"],
        stem_embeddings=embedding_out["stem_embeddings"],
    )
    ordered_df.to_parquet(metadata_path, index=False)

    manifest = {
        "run_label": run_label,
        "combined_name": combined_name,
        "included_splits": split_names,
        "source_run_dir": str(run_dir),
        "checkpoint_path": str(checkpoint_path),
        "output_dir": str(output_dir),
        "metadata_path": str(metadata_path),
        "embeddings_path": str(npz_path),
        "num_rows": int(len(ordered_df)),
        "embedding_dim": int(embedding_out["song_embeddings"].shape[1]),
        "device": str(device),
        "model_init_kwargs": model_init_kwargs,
        "source_run_manifest": source_manifest,
    }
    manifest_out_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print(f"saved: {output_dir}")
    print(f"rows: {len(ordered_df)} | dim: {embedding_out['song_embeddings'].shape[1]}")
    return manifest

## Build Train + Validation Embeddings For All Models

This is the main operation.

In [10]:
for split_name, split_path in CFG["split_paths"].items():
    if not split_path.exists():
        raise FileNotFoundError(f"Missing split file: {split_path}")

train_validation_summaries = []
for run_label in CFG["run_labels"]:
    print(f"\n=== Exporting train_validation for {run_label} ===")
    summary = export_combined_embeddings(
        run_label=run_label,
        split_names=["train", "validation"],
        output_root=CFG["train_validation_root"],
        combined_name="train_validation",
    )
    train_validation_summaries.append(summary)

index_path = CFG["train_validation_root"] / "index.json"
index_path.write_text(json.dumps(train_validation_summaries, indent=2), encoding="utf-8")
print(f"\nSaved index: {index_path}")
pd.DataFrame([
    {
        "run_label": item["run_label"],
        "combined_name": item["combined_name"],
        "num_rows": item["num_rows"],
        "embedding_dim": item["embedding_dim"],
        "output_dir": item["output_dir"],
    }
    for item in train_validation_summaries
])


=== Exporting train_validation for 04_Resnet18_contrastive_tags ===
saved: C:\Users\Juan\Documents\GitHub\dl-song-recommender\data\processed\model_runs\final_embeddings_train_validation\04_Resnet18_contrastive_tags
rows: 9549 | dim: 64

=== Exporting train_validation for 05_Resnet18_contrastive_tags_infonce ===
saved: C:\Users\Juan\Documents\GitHub\dl-song-recommender\data\processed\model_runs\final_embeddings_train_validation\05_Resnet18_contrastive_tags_infonce
rows: 9549 | dim: 64

=== Exporting train_validation for 06_Resnet18_contrastive_tags_audio_grounded_infonce ===
saved: C:\Users\Juan\Documents\GitHub\dl-song-recommender\data\processed\model_runs\final_embeddings_train_validation\06_Resnet18_contrastive_tags_audio_grounded_infonce
rows: 9549 | dim: 64

=== Exporting train_validation for 07_Resnet18_audio_centric_blended_teacher ===
saved: C:\Users\Juan\Documents\GitHub\dl-song-recommender\data\processed\model_runs\final_embeddings_train_validation\07_Resnet18_audio_centric_b

,run_label,combined_name,num_rows,embedding_dim,output_dir
0,04_Resnet18_contrastive_tags,train_validation,9549,64,C:\Users\Juan\Documents\GitHub\dl-song-recomme...
1,05_Resnet18_contrastive_tags_infonce,train_validation,9549,64,C:\Users\Juan\Documents\GitHub\dl-song-recomme...
2,06_Resnet18_contrastive_tags_audio_grounded_in...,train_validation,9549,64,C:\Users\Juan\Documents\GitHub\dl-song-recomme...
3,07_Resnet18_audio_centric_blended_teacher,train_validation,9549,64,C:\Users\Juan\Documents\GitHub\dl-song-recomme...


## Optional: Build Train + Validation + Test Embeddings

This is separate on purpose. It is basically the same operation as above, just with `test` added in.